In [128]:
!pip install bert-score transformers torch sentence-transformers numpy spacy

## Imports

In [129]:
import os
import nltk
import json
import torch
import spacy
import numpy as np
import torch.nn.functional as F

from bert_score import score
from transformers import logging

from sentence_transformers import SentenceTransformer
from transformers import BartTokenizer, BartForConditionalGeneration, AutoTokenizer, AutoModelForSequenceClassification


In [130]:
logging.set_verbosity_error()

## Contextual Semantic Metrics 

In [131]:
def calculate_bertscore(paragraph:str, rewrite:str, type="bert"):
    if type == "bert":
        model = "bert-base-uncased"
    if type == "scibert":
        model = "allenai/scibert_scivocab_uncased"

    P, R, F1 = score(
    cands= rewrite,
    refs= paragraph,
    lang="en",
    model_type=model,
    verbose=False,
    )
    
    return F1.mean().item()

In [132]:
def calculate_bartscore(paragraph:str, rewrite:str):
    model_name = "facebook/bart-large"
    tokenizer = BartTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name)
    model.eval()

    with torch.no_grad():
        inputs = tokenizer(paragraph, return_tensors="pt", truncation=True)
        labels = tokenizer(rewrite, return_tensors="pt", truncation=True)["input_ids"]

        outputs = model(**inputs, labels=labels)
        loss = outputs.loss  # cross-entropy
        
        bart_score = -loss.item()

    return bart_score

In [133]:
def calculate_cosine_similarity(paragraph:str, rewrite:str):
    model_name = "sentence-transformers/all-mpnet-base-v2"
    model = SentenceTransformer(model_name)
    
    embeddings = model.encode([paragraph, rewrite], show_progress_bar=False)
    
    vec1 = embeddings[0]
    vec2 = embeddings[1]
    
    cosine = np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )
    return float(cosine)

In [134]:
def calculate_semantic_metrics(paragraph, rewrite):
    bert_score = calculate_bertscore(paragraph, rewrite)
    #scibert_score = calculate_bertscore(paragraph, rewrite, "scibert")
    bart_score = calculate_bartscore(paragraph, rewrite)
    cosinus_similarity = calculate_cosine_similarity(paragraph[0], rewrite[0])
    return {"BertScore":bert_score,
            #"SciBertScore": scibert_score,
            "BartScore": bart_score,
            "Cos_Sim":cosinus_similarity}

## Factual metrics

In [135]:
tokenizer = AutoTokenizer.from_pretrained("manueldeprada/FactCC")
model = AutoModelForSequenceClassification.from_pretrained("manueldeprada/FactCC")

def factcc_score(paragraph, rewrite):

    inputs = tokenizer(
        paragraph,
        rewrite,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=1)

    return prediction.item()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 499.73it/s, Materializing param=classifier.weight]                                      


In [136]:
spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [137]:
def factacc_score(paragraph, rewrite):

    doc = nlp(rewrite)

    facts = []
    correct = 0

    for token in doc:
        if token.dep_ == "nsubj":
            subject = token.text
            verb = token.head.text

            for child in token.head.children:
                if child.dep_ == "dobj":
                    obj = child.text
                    facts.append((subject, verb, obj))

    for fact in facts:
        if all(word.lower() in paragraph.lower() for word in fact):
            correct += 1

    if len(facts) == 0:
        return 0

    return correct / len(facts)

In [138]:
def dae_score(paragraph, rewrite):

    doc = nlp(rewrite)

    arcs_total = 0
    arcs_supported = 0

    for token in doc:
        arcs_total += 1

        relation = f"{token.head.text} {token.dep_} {token.text}"

        if relation.lower() in paragraph.lower():
            arcs_supported += 1

    if arcs_total == 0:
        return 0

    return arcs_supported / arcs_total

In [139]:
def factacc_score(paragraph, rewrite):

    doc = nlp(rewrite)

    facts = []
    correct = 0

    for token in doc:
        if token.dep_ == "nsubj":
            subject = token.text
            verb = token.head.text

            for child in token.head.children:
                if child.dep_ == "dobj":
                    obj = child.text
                    facts.append((subject, verb, obj))

    for fact in facts:
        if all(word.lower() in paragraph.lower() for word in fact):
            correct += 1

    if len(facts) == 0:
        return 0

    return correct / len(facts)

In [140]:
def calculate_factual_metrics(paragraph, rewrite):
    fact_cc = factcc_score(paragraph, rewrite)
    fact_acc = factacc_score(paragraph, rewrite)
    dae = dae_score(paragraph, rewrite)

    return {"FactCC":fact_cc,
            "FactAcc": fact_acc,
            "DAE":dae}

In [141]:
def keys_update(dico,index=1):
    new_dico = {f"{k}{index}": v for k, v in dico.items()}
    return new_dico

In [142]:
def compute_all_metrics(data):
    for article in data:
        for i in range(1,4):

            semantic_metrics = calculate_semantic_metrics([article['Paragraphes']], [article[f"Rew{i}"]])
            semantic_metrics_updated = keys_update(semantic_metrics, i)
            article.update(semantic_metrics_updated)

            factual_metrics = calculate_factual_metrics(article['Paragraphes'], article[f"Rew{i}"])
            factual_metrics_updated = keys_update(factual_metrics,i)
            article.update(factual_metrics_updated)
    return data

## Code Tests

In [143]:
with open("../data/json/rewrites.json","r" , encoding='utf-8') as file:
    data = json.load(file)

In [144]:
filename = "../data/json/metrics.json"
if not os.path.exists(filename):
    with open(filename, "w", encoding="utf-8") as f:
        pass

In [145]:
all_metrics = compute_all_metrics(data)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 535.86it/s, Materializing param=pooler.dense.weight]                        


In [146]:
all_metrics

[{'Doi': 'https://doi.org/10.57896/2024-tal-65_3_3',
  'Paragraphes': 'A Hidden Markov Model (HMM) is a classical statistical model [32] that embodies a set of statistical assumptions concerning the generation of sample data (and similar data from a larger population). It is often used in NLP for tagging [33, 34]. It can be applied to speech recognition, part-of-speech tagging and various bioinformatics applications inter alia. Given an input sequence of n signs X = (x1, …, xn) and a possible output transliteration Y = (y1, …, yn), an HMM provides the probability p(X, Y). Thus, given a sequence of signs X = (x1, …, xn), we can choose the transliteration  that maximizes .',
  'Auteur': 'Delphine Battistelli, Farah Benamara, Viviana Patti',
  'Date de publication': '2025',
  'URL': 'https://aclanthology.org/2024.tal-3.4/',
  'Licence': 'CC BY 4.0',
  'Domaine': 'TALN',
  'Rew1': 'A Hidden Markov Model (HMM) is a classical statistical model that embodies a set of statistical assumptions c

In [147]:
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=4, ensure_ascii=False)